In [0]:
dbutils.widgets.text("catalog", "tpc_a_dev")
catalog = dbutils.widgets.get("catalog")

dbutils.widgets.text("job_run_id", "ADHOC")
job_run_id = dbutils.widgets.get("job_run_id")

# Pipeline run ID — unique per execution, used across all bronze tables
# import uuid
# pipeline_run_id = f"run_{spark.sparkContext.applicationId}_{uuid.uuid4().hex[:8]}"
# print(f"Pipeline run ID: {pipeline_run_id}")

In [0]:
from pyspark.sql import functions as F

##### Ingesting nation

In [0]:
(
    spark.read.format('csv')
    .option("header", True)
    .option("inferSchema", True)
    .load(f"/Volumes/{catalog}/bronze/raw/nation.csv")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_job_run_id", F.lit(job_run_id))
    .write.mode("overwrite")
    .saveAsTable(f"{catalog}.bronze.nation")
)

In [0]:
(
    spark.read.format('csv')
    .option("header", True)
    .option("inferSchema", True)
    .load(f"/Volumes/{catalog}/bronze/raw/region.csv")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_job_run_id", F.lit(job_run_id))
    .write.mode("overwrite")
    .saveAsTable(f"{catalog}.bronze.region")
)

In [0]:
spark.sql(f"Create table if not exists {catalog}.bronze.airline")

spark.sql(f"""
    Copy into {catalog}.bronze.airline
    From (
        Select *,
            _metadata.file_path as _source_file,
            current_timestamp() as _ingested_at,
            '{job_run_id}' as _job_run_id
        From '/Volumes/{catalog}/bronze/raw/airline/'
    )
    Fileformat = CSV
    Format_options ('header'='true', 'inferSchema'='true')
    Copy_options('mergeSchema'='true')
""")

spark.sql(f"select * from {catalog}.bronze.airline")

In [0]:
spark.sql(f"Create table if not exists {catalog}.bronze.route")

spark.sql(f"""
    Copy into {catalog}.bronze.route
    From (
        Select *,
            _metadata.file_path as _source_file,
            current_timestamp() as _ingested_at,
            '{job_run_id}' as _job_run_id
        From '/Volumes/{catalog}/bronze/raw/route/'
    )
    Fileformat = CSV
    Format_options ('header'='true', 'inferSchema'='true')
    Copy_options('mergeSchema'='true')
""")

In [0]:
spark.sql(f"Create table if not exists {catalog}.bronze.airline_route")

spark.sql(f"""
    Copy into {catalog}.bronze.airline_route
    From (
        Select *,
            _metadata.file_path as _source_file,
            current_timestamp() as _ingested_at,
            '{job_run_id}' as _job_run_id
        From '/Volumes/{catalog}/bronze/raw/airline_route/'
    )
    Fileformat = CSV
    Format_options ('header'='true', 'inferSchema'='true')
    Copy_options('mergeSchema'='true')
""")

In [0]:
spark.sql(f"Create table if not exists {catalog}.bronze.passenger")

spark.sql(f"""
    Copy into {catalog}.bronze.passenger
    From (
        Select *,
            _metadata.file_path as _source_file,
            current_timestamp() as _ingested_at,
            '{job_run_id}' as _job_run_id
        From '/Volumes/{catalog}/bronze/raw/passenger/'
    )
    Fileformat = CSV
    Format_options ('header'='true', 'inferSchema'='true')
    Copy_options('mergeSchema'='true')
""")

spark.sql(f"select * from {catalog}.bronze.passenger")

In [0]:
source_path = f"/Volumes/{catalog}/bronze/raw/booking"
checkpoint_path = f"/Volumes/{catalog}/bronze/pipeline_state/checkpoint/bookings"
schema_path = f"/Volumes/{catalog}/bronze/pipeline_state/schema/bookings"


df_booking_bronze = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option('cloudFiles.inferColumnTypes','true')
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .load(source_path)
)

(
    df_booking_bronze
    .withColumn("_source_filename", F.col("_metadata.file_name"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_job_run_id", F.lit(job_run_id))
    .writeStream.format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.booking")
)

spark.sql(f"""
COMMENT ON TABLE {catalog}.bronze.booking IS
'Raw booking data from csv files. Append-only, never modified.'
""")

In [0]:
source_path = f"/Volumes/{catalog}/bronze/raw/flight_booking"
checkpoint_path = f"/Volumes/{catalog}/bronze/pipeline_state/checkpoint/flight_bookings"
schema_path = f"/Volumes/{catalog}/bronze/pipeline_state/schema/flight_bookings"


df_booking_bronze = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option('cloudFiles.inferColumnTypes','true')
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .load(source_path)
)

(
    df_booking_bronze
    .withColumn("_source_filename", F.col("_metadata.file_name"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_job_run_id", F.lit(job_run_id))
    .writeStream.format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.flight_booking")
)

spark.sql(f"""
COMMENT ON TABLE {catalog}.bronze.flight_booking IS
'Raw flight_booking data from csv files. Append-only, never modified.'
""")

In [0]:
%sql
select count(*) from tpc_a_dev.bronze.flight_booking